# Tutorial 14 — Graph Neural Networks for Molecules: DRD2 Binding Prediction
**Author:** Himanshu Goel | [Website](https://hgoelgithub.github.io)

Molecules are natural graphs: atoms are nodes, bonds are edges. GNNs exploit this structure directly — unlike fingerprints, they learn their own features end-to-end. We build a Graph Convolutional Network (GCN) using PyTorch Geometric and train it to predict binding activity at the **Dopamine D2 receptor (DRD2)** — a key GPCR target for antipsychotic drugs.

Dataset: ~6,935 compounds from BindingDB with binary activity labels (Ki-derived). Split strategy: **Murcko scaffold-based 80/20 train/test** — ensures structurally novel scaffolds appear only in the test set, giving a more realistic evaluation than random splitting.

In [6]:
# PyTorch: deep learning framework used to build and train the GCN
!pip install torch -q
# PyTorch Geometric (PyG): extends PyTorch with graph neural network primitives
# (GCNConv, global pooling, DataLoader for graphs, etc.)
!pip install torch-geometric -q
# RDKit: cheminformatics library for parsing SMILES and extracting atom/bond info
# pandas/numpy: data handling; matplotlib: plotting training curves
!pip install rdkit pandas numpy matplotlib -q

In [7]:
# ── Imports ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, NNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from rdkit import Chem
from rdkit.Chem import rdchem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
import pandas as pd
import numpy as np

# ── Feature dimension constants ───────────────────────────────────────────────
NODE_DIM = 6   # atom feature vector length  (used by both GCN and MPNN)
EDGE_DIM = 6   # bond feature vector length  (used by MPNN's edge networks)

# Bond-type → integer index for one-hot encoding
BOND_TYPE_MAP = {
    rdchem.BondType.SINGLE:   0,
    rdchem.BondType.DOUBLE:   1,
    rdchem.BondType.TRIPLE:   2,
    rdchem.BondType.AROMATIC: 3,
}

# ── SMILES → PyG graph conversion ────────────────────────────────────────────
def mol_to_graph(smiles: str, label: float):
    """
    Converts a SMILES string into a PyTorch Geometric Data object.

    Node features (NODE_DIM = 6 per atom):
      0 - atomic number / 100        (normalised element identity)
      1 - degree / 6                 (number of bonds)
      2 - formal charge / 2          (charge state)
      3 - is_aromatic                (part of aromatic ring?)
      4 - total H count / 4          (implicit + explicit H)
      5 - is_in_ring                 (part of any ring?)

    Edge features (EDGE_DIM = 6 per bond, same for both directions):
      0-3 - one-hot bond type        (single / double / triple / aromatic)
        4 - is_conjugated
        5 - is_in_ring

    GCN ignores edge_attr; MPNN uses it to parameterise per-bond weight matrices.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # ── Node features ─────────────────────────────────────────────────────────
    x = []
    for atom in mol.GetAtoms():
        x.append([
            atom.GetAtomicNum() / 100.0,
            atom.GetDegree() / 6.0,
            atom.GetFormalCharge() / 2.0,
            float(atom.GetIsAromatic()),
            atom.GetTotalNumHs() / 4.0,
            float(atom.IsInRing()),
        ])
    x = torch.tensor(x, dtype=torch.float)

    # ── Edge index + bond features ────────────────────────────────────────────
    # Each bond is stored twice (i→j and j→i) for undirected message passing.
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = BOND_TYPE_MAP.get(bond.GetBondType(), 0)
        feat = [
            float(bt == 0),                    # single bond
            float(bt == 1),                    # double bond
            float(bt == 2),                    # triple bond
            float(bt == 3),                    # aromatic bond
            float(bond.GetIsConjugated()),      # conjugated system
            float(bond.IsInRing()),             # in ring
        ]
        edge_index += [[i, j], [j, i]]
        edge_attr  += [feat, feat]             # same features for both directions

    if not edge_index:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float))

# ── Load DRD2 dataset from BindingDB ─────────────────────────────────────────
# DRD2 (Dopamine D2 receptor) — GPCR target central to antipsychotic pharmacology.
# Activity: 1 = active binder (Ki below threshold), 0 = inactive.
df = pd.read_csv("D2_combined_Ki.csv")
print(f"Loaded {len(df)} rows  |  actives: {(df.Activity==1).sum()}  inactives: {(df.Activity==0).sum()}")

graphs, valid_smiles = [], []
for smi, lbl in zip(df["SMILES"].tolist(), df["Activity"].tolist()):
    g = mol_to_graph(smi, float(lbl))
    if g is not None:
        graphs.append(g)
        valid_smiles.append(smi)

print(f"Valid graphs    : {len(graphs)}")
print(f"Node dim        : {NODE_DIM}   Edge dim: {EDGE_DIM}")

# ── Scaffold-based 80/20 split ────────────────────────────────────────────────
# Murcko scaffold = the core ring system (side chains stripped off).
# All analogues sharing a scaffold go to the same split — no structural leakage.
def get_scaffold(smi: str) -> str:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

scaffold_to_idxs: dict = defaultdict(list)
for idx, smi in enumerate(valid_smiles):
    scaffold_to_idxs[get_scaffold(smi)].append(idx)

scaffold_sets = sorted(scaffold_to_idxs.values(), key=len, reverse=True)
train_cutoff  = int(0.8 * len(graphs))
train_idx, test_idx = [], []

for scaffold_idxs in scaffold_sets:
    if len(train_idx) + len(scaffold_idxs) <= train_cutoff:
        train_idx.extend(scaffold_idxs)
    else:
        test_idx.extend(scaffold_idxs)

train_data = [graphs[i] for i in train_idx]
test_data  = [graphs[i] for i in test_idx]

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64)

n_pos = sum(g.y.item() == 1 for g in graphs)
n_neg = len(graphs) - n_pos
print(f"Train / test    : {len(train_data)} / {len(test_data)}")
print(f"Class balance   : {n_neg} inactives / {n_pos} actives  →  pos_weight = {n_neg/n_pos:.4f}")

Loaded 6935 rows  |  actives: 6600  inactives: 335
Valid graphs    : 6935
Node dim        : 6   Edge dim: 6
Train / test    : 5548 / 1387
Class balance   : 335 inactives / 6600 actives  →  pos_weight = 0.0508


In [8]:
# ── GCN Model ─────────────────────────────────────────────────────────────────
class MolGCN(nn.Module):
    """
    Three-layer Graph Convolutional Network for binary molecule classification.

    Architecture:
      GCNConv(6→64) → BN → ReLU → Dropout
      GCNConv(64→64) → BN → ReLU
      GCNConv(64→64) → ReLU
      global_mean_pool  →  Linear(64→1)  [raw logit for BCEWithLogitsLoss]

    GCNConv (Kipf & Welling 2017):  H' = σ( D̂^{-½} Â D̂^{-½} H W )
    Â = A + I adds self-loops; D̂ is the normalised degree matrix.
    Each atom aggregates neighbour features weighted by graph connectivity.
    Edge features (edge_attr) are present in the Data objects but unused here —
    GCN uses a single shared weight matrix, not per-bond transformations.
    """
    def __init__(self, in_ch=NODE_DIM, hidden=64, out_ch=1):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.fc    = nn.Linear(hidden, out_ch)
        self.drop  = nn.Dropout(0.3)

    def forward(self, x, edge_index, batch, edge_attr=None):
        # edge_attr accepted but unused — keeps the same call signature as MPNN
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.drop(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.fc(x).squeeze()

# ── Instantiate GCN, optimiser, and shared loss ───────────────────────────────
gcn_model  = MolGCN()
gcn_opt    = torch.optim.Adam(gcn_model.parameters(), lr=5e-3, weight_decay=1e-4)

# BCEWithLogitsLoss = numerically stable fused sigmoid + BCE.
# pos_weight ≈ 0.051 down-weights the majority active class so both classes
# contribute roughly equally to the gradient despite the 95:5 imbalance.
pos_weight = torch.tensor([n_neg / n_pos])
loss_fn    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Training loop ─────────────────────────────────────────────────────────────
gcn_losses = []
for epoch in range(80):
    gcn_model.train()
    epoch_loss = 0
    for batch in train_loader:
        gcn_opt.zero_grad()
        pred = gcn_model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(pred, batch.y.squeeze())
        loss.backward()
        gcn_opt.step()
        epoch_loss += loss.item()
    gcn_losses.append(epoch_loss / len(train_loader))
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}  GCN loss={gcn_losses[-1]:.4f}")

Epoch   0  GCN loss=0.0649
Epoch  20  GCN loss=0.0545
Epoch  40  GCN loss=0.0516
Epoch  60  GCN loss=0.0492


In [ ]:
# ── MPNN Model ────────────────────────────────────────────────────────────────
class MolMPNN(nn.Module):
    """
    Message Passing Neural Network (Gilmer et al. 2017) for binary classification.

    Key difference from GCN: bond features drive a *per-edge weight matrix*.
    NNConv computes:
        m_ij = Θ(e_ij) · h_j          where Θ(e) is a small MLP applied to bond features
        h_i' = h_i + mean_{j∈N(i)} m_ij

    This lets the model treat single, double, aromatic bonds differently —
    GCN cannot, because it uses a single shared weight matrix for all edges.

    Architecture:
      NNConv(6→64, edge_net_1) → BN → ReLU → Dropout   [hop 1: 6 atom feats → 64]
      NNConv(64→64, edge_net_2) → BN → ReLU              [hop 2: refine with bond info]
      global_mean_pool  →  Linear(64→1)                  [graph-level classifier]

    Edge networks:  edge_dim=6  →  32  →  (in_ch × out_ch)  [flattened weight matrix]
    """
    def __init__(self, node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden=64, out_ch=1):
        super().__init__()
        # edge_net output size = in_channels × out_channels (flattened weight matrix)
        self.conv1 = NNConv(
            node_dim, hidden,
            nn.Sequential(
                nn.Linear(edge_dim, 32), nn.ReLU(),
                nn.Linear(32, node_dim * hidden),   # 6 × 64 = 384
            ),
            aggr='mean',
        )
        self.conv2 = NNConv(
            hidden, hidden,
            nn.Sequential(
                nn.Linear(edge_dim, 32), nn.ReLU(),
                nn.Linear(32, hidden * hidden),     # 64 × 64 = 4096
            ),
            aggr='mean',
        )
        self.bn1  = nn.BatchNorm1d(hidden)
        self.bn2  = nn.BatchNorm1d(hidden)
        self.fc   = nn.Linear(hidden, out_ch)
        self.drop = nn.Dropout(0.3)

    def forward(self, x, edge_index, batch, edge_attr=None):
        # edge_attr carries the 6-d bond features; NNConv maps them to weight matrices
        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = self.drop(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index, edge_attr)))
        x = global_mean_pool(x, batch)
        return self.fc(x).squeeze()

# ── Train MPNN ────────────────────────────────────────────────────────────────
mpnn_model = MolMPNN()
mpnn_opt   = torch.optim.Adam(mpnn_model.parameters(), lr=5e-3, weight_decay=1e-4)
# Reuse the same class-balanced loss defined in the GCN cell
mpnn_losses = []

for epoch in range(80):
    mpnn_model.train()
    epoch_loss = 0
    for batch in train_loader:
        mpnn_opt.zero_grad()
        pred = mpnn_model(batch.x, batch.edge_index, batch.batch, batch.edge_attr)
        loss = loss_fn(pred, batch.y.squeeze())
        loss.backward()
        mpnn_opt.step()
        epoch_loss += loss.item()
    mpnn_losses.append(epoch_loss / len(train_loader))
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}  MPNN loss={mpnn_losses[-1]:.4f}")

# ── Training curve comparison ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gcn_losses,  color="#1565c0", lw=2, label="GCN")
ax.plot(mpnn_losses, color="#c62828", lw=2, label="MPNN")
ax.set_xlabel("Epoch"); ax.set_ylabel("Weighted BCE Loss")
ax.set_title("Training Curves — GCN vs MPNN  (DRD2 scaffold split)")
ax.legend(); plt.tight_layout()
plt.savefig("gcn_vs_mpnn_loss.png", dpi=150); plt.show()

In [ ]:
# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate(model, loader):
    """
    Run inference on a DataLoader and return a metrics dict.

    Computed metrics
    ----------------
    accuracy   : (TP+TN) / N
    precision  : TP / (TP+FP)   — of all predicted actives, how many are truly active
    recall     : TP / (TP+FN)   — fraction of true actives that were found
    f1         : harmonic mean of precision and recall
    mcc        : Matthews Correlation Coefficient
                   = (TP·TN − FP·FN) / √[(TP+FP)(TP+FN)(TN+FP)(TN+FN)]
                 Range: −1 (worst) … 0 (random) … +1 (perfect).
                 Unlike accuracy or F1, MCC accounts for all four quadrants of
                 the confusion matrix and is reliable even under class imbalance.
    roc_auc    : area under the ROC curve (O(n log n), no sklearn required)
    """
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch, batch.edge_attr)
            if logits.dim() == 0:
                logits = logits.unsqueeze(0)
            all_probs.extend(torch.sigmoid(logits).tolist())
            all_labels.extend(batch.y.squeeze().tolist())

    probs  = np.array(all_probs)
    labels = np.array(all_labels, dtype=int)
    preds  = (probs >= 0.5).astype(int)

    tp = int(np.sum((preds == 1) & (labels == 1)))
    tn = int(np.sum((preds == 0) & (labels == 0)))
    fp = int(np.sum((preds == 1) & (labels == 0)))
    fn = int(np.sum((preds == 0) & (labels == 1)))

    acc  = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    # Matthews Correlation Coefficient
    # Denominator can be 0 only when an entire row/column of the confusion matrix is zero.
    denom = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc   = (tp * tn - fp * fn) / denom if denom > 0 else 0.0

    # ROC-AUC: O(n log n) via cumulative sort — no sklearn needed
    n_pos = int(labels.sum()); n_neg = len(labels) - n_pos
    sort_idx     = np.argsort(probs)[::-1]           # descending probability
    sorted_lbl   = labels[sort_idx]
    tpr_pts = np.concatenate([[0.0], np.cumsum(sorted_lbl == 1) / n_pos,  [1.0]])
    fpr_pts = np.concatenate([[0.0], np.cumsum(sorted_lbl == 0) / n_neg,  [1.0]])
    # Sort by FPR to handle probability ties before numerical integration
    order   = np.argsort(fpr_pts)
    roc_auc = float(np.trapz(tpr_pts[order], fpr_pts[order]))

    return {
        "accuracy": acc, "precision": prec, "recall": rec,
        "f1": f1, "mcc": mcc, "roc_auc": roc_auc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "tpr": tpr_pts[order], "fpr": fpr_pts[order],
    }

# ── Evaluate both models ──────────────────────────────────────────────────────
gcn_res  = evaluate(gcn_model,  test_loader)
mpnn_res = evaluate(mpnn_model, test_loader)

# ── Side-by-side metrics table ────────────────────────────────────────────────
METRIC_LABELS = [
    ("accuracy",  "Accuracy"),
    ("precision", "Precision"),
    ("recall",    "Recall"),
    ("f1",        "F1"),
    ("mcc",       "MCC"),        # ← Matthews Correlation Coefficient
    ("roc_auc",   "ROC-AUC"),
]
n_test = len(gcn_res["tpr"]) - 2   # subtract the two sentinel points
print(f"Test set: {n_test} molecules\n")
print(f"{'Metric':<12}  {'GCN':>8}  {'MPNN':>8}")
print("─" * 32)
for key, lbl in METRIC_LABELS:
    print(f"{lbl:<12}  {gcn_res[key]:>8.4f}  {mpnn_res[key]:>8.4f}")

# ── Confusion matrices ────────────────────────────────────────────────────────
print()
for name, res in [("GCN", gcn_res), ("MPNN", mpnn_res)]:
    print(f"{name} confusion matrix:")
    print(f"                  Pred active   Pred inactive")
    print(f"  True active       {res['tp']:6d}          {res['fn']:6d}")
    print(f"  True inactive     {res['fp']:6d}          {res['tn']:6d}")
    print()

# ── ROC curve — both models on the same axes ──────────────────────────────────
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(gcn_res["fpr"],  gcn_res["tpr"],
        color="#1565c0", lw=2, label=f"GCN  (AUC={gcn_res['roc_auc']:.3f}, MCC={gcn_res['mcc']:.3f})")
ax.plot(mpnn_res["fpr"], mpnn_res["tpr"],
        color="#c62828", lw=2, label=f"MPNN (AUC={mpnn_res['roc_auc']:.3f}, MCC={mpnn_res['mcc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — DRD2 Binding Prediction (scaffold split)")
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig("gcn_roc.png", dpi=150); plt.show()

## Key takeaways

**Architectures**
- **GCN** (Kipf & Welling 2017): spectral convolution with a single shared weight matrix — bond type is ignored
- **MPNN** (Gilmer et al. 2017): `NNConv` learns a *per-edge* weight matrix from bond features, letting the model distinguish single, double, and aromatic bonds
- Both use `global_mean_pool` to aggregate atom embeddings into a fixed-size graph vector for classification

**Data pipeline**
- `mol_to_graph` now produces both node features (6-d) and bond features (6-d one-hot type + conjugated + in-ring); GCN silently ignores `edge_attr`, MPNN uses it
- **Scaffold splitting** (Murcko) assigns all analogues of a scaffold to the same split, preventing structural leakage between train and test

**Evaluation under class imbalance (95 % actives)**
- Raw accuracy is uninformative — a trivial "predict all active" rule scores ≈ 95 %
- **MCC** (Matthews Correlation Coefficient) is the most reliable single number for imbalanced binary classification: it incorporates all four confusion matrix cells and equals 0 for any constant predictor
- **ROC-AUC** measures rank-ordering quality independently of the decision threshold
- `BCEWithLogitsLoss(pos_weight=n_neg/n_pos)` re-balances gradient contributions without any resampling